# yaml 설정 그대로의 BSRNN — 모든 축에서 컴파일 시간과 step 속도

**묻는 것** — [confs/bsrnn_ecapa_FiLM.yaml](../examples/librimix/tse/v2/confs/bsrnn_ecapa_FiLM.yaml) 의
설정을 **그대로 둔 채**, 돌리는 방식만 바꿨을 때 컴파일 비용과 스텝 속도가 어떻게 달라지는가.

모델 구조는 **건드리지 않음** — 구조를 바꾸는 쪽은
[bsrnn_compile_cost_by_structure.ipynb](bsrnn_compile_cost_by_structure.ipynb) 가 맡음.

## 축 5개

| 축 | 값 | 뜻 |
|---|---|---|
| **`scope`** | `eager` · `separator` · `separator&spk` · `model` | `.compile()` 을 **어디에** 거는가 |
| **`dynamic`** | `False` · `True` | `.compile(dynamic=)`. `eager` 에서는 뜻이 없어 `-` 로 둠 |
| **`b`** | 1 · 2 · 4 · 8 · 16 | 모델이 보는 배치. yaml `batch_size: 8` 은 화자 2명으로 펴져 **`b=16`** 이 됨 |
| **`prec`** | `fp32` · `bf16-mixed` · `fp16-mixed` | `autocast` dtype 과 `GradScaler` 사용 여부 |
| **`tf32_lstm`** | `True` · `False` | `torch.backends.cudnn.allow_tf32`. **LSTM 은 cuDNN 경로**라 여기에 걸림 |

## 0. 준비 — 측정 코드는 `_bench/bench_common.py` 에 있음

이 노트북은 **측정을 직접 하지 않음.** 워커가 GPU 여러 장에서 재어
[_runs/](_runs/) 에 쌓아 둔 결과를 **읽어서 표만 그림.**

| 무엇 | 어디 |
|---|---|
| 측정 코드 (모델 · 로더 모사 · `bench()` · 조건 목록) | [_bench/bench_common.py](_bench/bench_common.py) |
| 워커 | [_bench/worker.py](_bench/worker.py) |
| 결과 | `_runs/<tag>/<조건해시>.json` — 조건 하나에 파일 하나 |

워커를 띄우는 법 (GPU 3장 기준):

```bash
cd wesep/notebooks/_bench
CUDA_VISIBLE_DEVICES=0 python worker.py --worker 0 --n-workers 3
CUDA_VISIBLE_DEVICES=1 python worker.py --worker 1 --n-workers 3
CUDA_VISIBLE_DEVICES=3 python worker.py --worker 2 --n-workers 3
```

`scope`·`dynamic` 만 다른 조건은 **한 묶음으로 같은 GPU** 에 감 —
`speedup` 의 기준이 되는 `eager` 와 떨어지면 비교가 깨지기 때문임.

## 측정 규약

| 항목 | 값 |
|---|---|
| 웜업 | **10스텝** — 등록 길이가 매 스텝 달라지므로 재컴파일을 여기서 끝냄 |
| 측정 | **50스텝**. **`ms_step_median`**(중앙값)과 **`ms_step_mean`**(평균)을 함께 남김 |
| 컴파일 시간 | **첫 스텝**의 벽시계 시간. 조건마다 `force_disable_caches = True` 로 **콜드** 보장 |
| 데이터 | **wesep 데이터로더 모사** — `wav_mix` 는 고정, 등록 발화는 배치 최소 길이로 잘림 |

| 표 읽는 법 | |
|---|---|
| 왼쪽 | **통제변인** — `scope` · `dynamic` · `b` · `t` · `prec` · `tf32_lstm` · `num_repeat` · `feature_dim` · `allow_rnn` · `gpu` |
| 오른쪽 | **측정값** — `compile_s` · `ms_step_median` · `ms_step_mean` · `peak_GiB` · `graph_break` · `dynamo_frames` · `recompile_in_measure` · `status` |

`dynamo_frames` 는 dynamo 가 컴파일한 **프레임 수**, `graph_break` 는 그래프가 **끊긴 횟수**로 서로 다른 값임.
**`recompile_in_measure` 가 0 이 아니면 웜업이 모자라 측정 구간에 재컴파일이 섞인 것**이므로 그 행은 믿지 말 것.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "_bench"))
import bench_common as bc

display(bc.env_table())
display(bc.conf_table())

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/kaldiio/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/s3prl/upstream/byol_s/byol_a/common.py:20: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")
ESPnet is not installed, cannot use espnet_hubert upstream


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,항목,값,비고
0,torch,2.7.1+cu128,cuda 12.8
1,DEVICE,cuda,모든 측정이 이 장치에서 돎
2,GPU,NVIDIA GeForce RTX 3090,23.6 GiB
3,CUDA_VISIBLE_DEVICES,0,이 GPU 에 다른 프로세스가 있으면 측정이 오염됨
4,wesep 루트,/workspace/git_clone/SD-FiLM/wesep,sys.path 에 넣음
5,결과 CSV,/workspace/git_clone/SD-FiLM/wesep/notebooks/_...,조건 하나가 한 줄


,항목,값,뜻
0,dataloader_args.batch_size,8,혼합 개수. 모델이 보는 배치는 이것의 2배
1,모델 배치 (실효),16,tse_collate_fn 이 편 뒤의 b
2,dataset_args.chunk_len,48000,입력 길이 t (샘플) = 3.0 초
3,model_args.num_repeat,6,BSNet 반복 수. LSTM 개수는 이것의 2배
4,model_args.feature_dim,128,밴드당 채널 c
5,fbank num_mel_bins,80,등록 발화 fbank 의 f
6,clip_grad,5.0,executor.py 가 매 스텝 부름
7,ECAPA 가중치 존재,True,/workspace/git_clone/SD-FiLM/wesep/examples/li...


### 0.1 wesep 데이터로더 모사 — 왜 등록 발화만 길이가 변하나

[tse_collate_fn](../wesep/dataset/dataset.py#L206-L264) 이 하는 일 중 shape 에 관계된 것만 옮겼음.

| # | 로더가 하는 일 | 결과 |
|---|---|---|
| 1 | 샘플마다 화자 2명분 `wav_mix` 를 복제 | 모델 배치 `b` = yaml `batch_size` × 2 |
| 2 | `wav_mix` 는 `chunk_len` 으로 이미 잘려 있음 | **separator 입력은 항상 고정** |
| 3 | 등록 발화는 화자마다 길이가 달라 `mode="min"` 이 **배치 최소 길이로 잘라냄** | **`spk_model` 입력은 배치마다 달라짐** |

그래서 `separator` 는 정적 shape, `spk_model`(ECAPA)은 동적 shape 을 받음 — 컴파일 조건이 서로 반대임.

**개별 발화 길이 분포는 근사이고 미검증임.** 실측한 것은 `b=16` 일 때 `min` 의 분포(299~1005 · 중앙값 396)뿐이라,
개별 길이를 `[299, 1950]` 균등으로 잡아 그 `min` 이 실측과 맞도록 역산했음.

In [2]:
display(bc.enroll_probe_table())

,b,등록 T 최소,등록 T 중앙값,등록 T 최대,고유 개수(60스텝)
0,1,346,1105,1930,60
1,2,315,793,1794,58
2,4,302,555,1700,60
3,8,300,430,907,60
4,16,300,362,761,54


### 0.2 측정 진행 상황

워커가 도는 중에도 이 셀만 다시 돌리면 어디까지 갔는지 보임.
`남음` 이 0 이 아닌 태그의 표는 **아직 일부만 그려진 것**임.

In [3]:
display(bc.progress_table())

,tag,조건 수,측정됨,남음,진행
0,structure-depth,15,15,0,100 %
1,structure-width,6,6,0,100 %
2,structure-who,5,5,0,100 %
3,structure-allow-rnn,4,4,0,100 %
4,axes-base,7,7,0,100 %
5,axes-single,10,10,0,100 %
6,axes-full,210,210,0,100 %
7,dynamic-none,4,4,0,100 %
8,dynamic-none-axes,12,12,0,100 %
9,dynamic-none-scopes,24,24,0,100 %


---

## 1. 조건 개수

`eager` 는 `dynamic` 이 뜻이 없어 한 벌만 돎 — 그래서 `scope × dynamic` 은 8이 아니라 **7가지**임.

In [4]:
import pandas as pd

display(pd.DataFrame([
    {"축": "scope x dynamic",
     "값": " / ".join(f"{s}:{d}" for s, d in bc.SCOPE_DYNAMIC), "가짓수": len(bc.SCOPE_DYNAMIC)},
    {"축": "b", "값": str(bc.BATCHES), "가짓수": len(bc.BATCHES)},
    {"축": "prec", "값": str(bc.PRECS), "가짓수": len(bc.PRECS)},
    {"축": "tf32_lstm", "값": str(bc.TF32), "가짓수": len(bc.TF32)},
]))

display(pd.DataFrame([{"tag": t, "조건 수": len(bc.conditions(t))}
                      for t in ("axes-base", "axes-single", "axes-full")]))

,축,값,가짓수
0,scope x dynamic,eager:False / separator:False / separator:True...,7
1,b,"(1, 2, 4, 8, 16)",5
2,prec,"('fp32', 'bf16-mixed', 'fp16-mixed')",3
3,tf32_lstm,"(True, False)",2


,tag,조건 수
0,axes-base,7
1,axes-single,10
2,axes-full,210


---

## 2. 기준점 — yaml 그대로

다른 모든 절이 이 값과 비교됨. `b=16` · `t=48000` · `fp16-mixed` · `tf32_lstm=True` 는
[현행 config](../examples/librimix/tse/v2/confs/bsrnn_ecapa_FiLM.yaml) 그대로임.

In [5]:
rows_base = bc.run_or_load("axes-base")
display(bc.as_table(rows_base))

bf = pd.DataFrame(rows_base)
if len(bf):
    e = bf[bf["scope"] == "eager"]["ms_step_median"]
    if len(e):
        bf["speedup"] = (float(e.iloc[0]) / bf["ms_step_median"]).round(3)
        display(bf[["scope", "dynamic", "compile_s", "ms_step_median", "ms_step_mean", "speedup",
                    "graph_break", "dynamo_frames", "recompile_in_measure", "peak_GiB", "status"]])

,scope,dynamic,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,eager,-,0.0,626.3,626.8,19.40,0,0,0,ok,NaN
1,separator,False,8.8,628.2,631.0,19.38,3,6,0,ok,NaN
2,separator,True,14.7,614.8,620.2,19.38,4,10,0,ok,NaN
3,separator&spk,False,24.7,618.5,7913.3,19.34,3,39,22,ok,NaN
4,separator&spk,True,43.3,613.4,616.0,19.38,4,11,0,ok,NaN
5,model,False,125.0,599.9,20172.2,19.35,35,40,22,ok,NaN
6,model,True,225.6,576.2,578.0,19.44,5,12,0,ok,NaN


,scope,dynamic,compile_s,ms_step_median,ms_step_mean,speedup,graph_break,dynamo_frames,recompile_in_measure,peak_GiB,status
0,eager,-,0.0,626.3,626.8,1.000,0,0,0,19.40,ok
1,separator,False,8.8,628.2,631.0,0.997,3,6,0,19.38,ok
2,separator,True,14.7,614.8,620.2,1.019,4,10,0,19.38,ok
3,separator&spk,False,24.7,618.5,7913.3,1.013,3,39,22,19.34,ok
4,separator&spk,True,43.3,613.4,616.0,1.021,4,11,0,19.38,ok
5,model,False,125.0,599.9,20172.2,1.044,35,40,22,19.35,ok
6,model,True,225.6,576.2,578.0,1.087,5,12,0,19.44,ok


---

## 3. 축 하나씩 — 나머지는 yaml 기본값 고정

각 축이 **단독으로** 무엇을 바꾸는지 봄. 전 조합(4절)보다 훨씬 싸므로 여기서 먼저 경향을 잡음.
`scope × dynamic` 은 2절이 이미 다 돌았으므로 여기서는 나머지 세 축만 돎.

In [6]:
rows_single = bc.run_or_load("axes-single")
display(bc.as_table(rows_single, keep=("b", "prec", "tf32_lstm")))

,scope,b,prec,tf32_lstm,gpu,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,separator&spk,1,fp16-mixed,True,0,NaN,NaN,NaN,NaN,0,0,0,ERROR,TorchRuntimeError: Dynamo failed to run FX nod...
1,separator&spk,2,fp16-mixed,True,1,39.2,424.6,431.1,2.77,4,11,0,ok,NaN
2,separator&spk,4,fp16-mixed,True,3,39.8,502.7,506.3,5.13,4,11,0,ok,NaN
3,separator&spk,8,fp16-mixed,True,0,48.7,460.5,466.1,9.94,4,11,0,ok,NaN
4,separator&spk,16,fp16-mixed,True,0,47.4,615.9,620.0,19.38,4,11,0,ok,NaN
5,separator&spk,16,fp32,True,1,NaN,NaN,NaN,NaN,0,0,0,OOM,NaN
6,separator&spk,16,bf16-mixed,True,1,38.0,624.8,626.8,19.38,4,11,0,ok,NaN
7,separator&spk,16,fp16-mixed,True,0,47.4,615.9,620.0,19.38,4,11,0,ok,NaN
8,separator&spk,16,fp16-mixed,True,0,47.4,615.9,620.0,19.38,4,11,0,ok,NaN
9,separator&spk,16,fp16-mixed,False,3,40.3,610.4,611.4,19.38,4,11,0,ok,NaN


---

## 4. 전 조합

`scope×dynamic`(7) × `b`(5) × `prec`(3) × `tf32_lstm`(2).
워커가 GPU 여러 장에 나눠 재 둔 것을 여기서는 **읽기만** 함.

In [7]:
rows_full = bc.run_or_load("axes-full")
display(bc.as_table(rows_full, keep=("b", "prec", "tf32_lstm")))

,scope,dynamic,b,prec,tf32_lstm,gpu,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,eager,-,1,fp32,True,3,NaN,NaN,NaN,NaN,0,0,0,ERROR,ValueError: Expected more than 1 value per cha...
1,eager,-,1,fp32,False,1,NaN,NaN,NaN,NaN,0,0,0,ERROR,ValueError: Expected more than 1 value per cha...
2,eager,-,1,bf16-mixed,True,1,NaN,NaN,NaN,NaN,0,0,0,ERROR,ValueError: Expected more than 1 value per cha...
3,eager,-,1,bf16-mixed,False,0,NaN,NaN,NaN,NaN,0,0,0,ERROR,ValueError: Expected more than 1 value per cha...
4,eager,-,1,fp16-mixed,True,0,NaN,NaN,NaN,NaN,0,0,0,ERROR,ValueError: Expected more than 1 value per cha...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,model,True,16,fp32,False,3,NaN,NaN,NaN,NaN,0,0,0,OOM,OutOfMemoryError: CUDA out of memory. Tried to...
206,model,True,16,bf16-mixed,True,3,221.9,594.1,593.9,19.39,5,12,0,ok,NaN
207,model,True,16,bf16-mixed,False,3,215.7,592.6,592.8,19.39,5,12,0,ok,NaN
208,model,True,16,fp16-mixed,True,0,214.2,569.8,572.8,19.40,5,12,0,ok,NaN


---

## 5. 종합

`speedup` 은 **같은 `(b, t, prec, tf32_lstm)` 의 `eager`** 를 기준으로 한 비임 —
조건이 다르면 절대 ms 를 비교할 수 없어 그렇게 묶음.

In [8]:
df_all = pd.DataFrame(rows_base + rows_single + rows_full).drop_duplicates(
    subset=bc.LEFT[:-1], keep="first")
ok = df_all[df_all["status"] == "ok"].copy()

key = ["b", "t", "prec", "tf32_lstm"]
base_ms = ok[ok["scope"] == "eager"].groupby(key)["ms_step_median"].median().rename("eager_ms")
ok = ok.merge(base_ms, left_on=key, right_index=True, how="left")
ok["speedup"] = (ok["eager_ms"] / ok["ms_step_median"]).round(3)

display(ok[["scope", "dynamic", "b", "prec", "tf32_lstm", "gpu",
            "compile_s", "ms_step_median", "ms_step_mean", "eager_ms", "speedup",
            "peak_GiB", "graph_break", "dynamo_frames", "status"]]
        .sort_values(["prec", "b", "scope", "dynamic"]))

for axis in ("scope", "dynamic", "b", "prec", "tf32_lstm"):
    display(ok.groupby(axis).agg(조건수=("ms_step_median", "size"),
                                 ms_step_median_중앙값=("ms_step_median", "median"),
                                 compile_s_중앙값=("compile_s", "median"),
                                 speedup_중앙값=("speedup", "median")).round(2).reset_index())

oom = df_all[df_all["status"] == "OOM"]
display(oom[["scope", "dynamic", "b", "prec", "tf32_lstm"]] if len(oom)
        else pd.DataFrame([{"OOM": "없음 — 모든 조건 측정됨"}]))

bad = df_all[df_all["recompile_in_measure"] != 0]
display(bad[["scope", "dynamic", "b", "prec", "tf32_lstm",
             "recompile_in_measure", "dynamo_frames"]] if len(bad)
        else pd.DataFrame([{"웜업 10스텝 판정": "충분 — 측정 구간 재컴파일 0건"}]))

,scope,dynamic,b,prec,tf32_lstm,gpu,compile_s,ms_step_median,ms_step_mean,eager_ms,speedup,peak_GiB,graph_break,dynamo_frames,status
25,eager,-,2,bf16-mixed,True,1,0.0,505.9,505.9,505.9,1.000,2.75,0,0,ok
26,eager,-,2,bf16-mixed,False,0,0.0,424.5,430.0,424.5,1.000,2.75,0,0,ok
175,model,False,2,bf16-mixed,True,1,148.0,465.8,25495.6,505.9,1.086,2.77,35,40,ok
176,model,False,2,bf16-mixed,False,0,159.2,425.9,26462.4,424.5,0.997,2.88,35,40,ok
205,model,True,2,bf16-mixed,True,1,539.0,401.9,412.5,505.9,1.259,2.76,5,12,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,separator,True,8,fp32,False,1,15.1,713.8,718.1,785.3,1.100,18.80,4,10,ok
125,separator&spk,False,8,fp32,True,3,28.4,601.5,8524.0,611.5,1.017,18.80,3,39,ok
126,separator&spk,False,8,fp32,False,1,28.9,734.1,8815.3,785.3,1.070,18.80,3,39,ok
155,separator&spk,True,8,fp32,True,3,46.6,585.8,590.8,611.5,1.044,18.80,4,11,ok


,scope,조건수,ms_step_median_중앙값,compile_s_중앙값,speedup_중앙값
0,eager,22,461.55,0.00,1.00
1,model,44,461.90,202.25,1.07
2,separator,44,434.10,13.40,1.03
3,separator&spk,44,462.10,39.05,1.02


,dynamic,조건수,ms_step_median_중앙값,compile_s_중앙값,speedup_중앙값
0,-,22,461.55,0.00,1.00
1,False,66,470.20,29.35,1.02
2,True,66,432.05,48.70,1.04


,b,조건수,ms_step_median_중앙값,compile_s_중앙값,speedup_중앙값
0,2,42,418.10,24.10,1.00
1,4,42,436.60,26.65,1.03
2,8,42,476.05,30.95,1.03
3,16,28,617.05,34.30,1.03


,prec,조건수,ms_step_median_중앙값,compile_s_중앙값,speedup_중앙값
0,bf16-mixed,56,451.8,32.10,1.06
1,fp16-mixed,56,445.5,31.05,1.00
2,fp32,42,473.5,24.45,1.03


,tf32_lstm,조건수,ms_step_median_중앙값,compile_s_중앙값,speedup_중앙값
0,False,77,456.1,29.8,1.03
1,True,77,447.6,28.4,1.02


,scope,dynamic,b,prec,tf32_lstm
12,separator&spk,True,16,fp32,True
41,eager,-,16,fp32,True
42,eager,-,16,fp32,False
71,separator,False,16,fp32,True
72,separator,False,16,fp32,False
101,separator,True,16,fp32,True
102,separator,True,16,fp32,False
131,separator&spk,False,16,fp32,True
132,separator&spk,False,16,fp32,False
162,separator&spk,True,16,fp32,False


,scope,dynamic,b,prec,tf32_lstm,recompile_in_measure,dynamo_frames
3,separator&spk,False,16,fp16-mixed,True,22,39
5,model,False,16,fp16-mixed,True,22,40
113,separator&spk,False,2,fp32,True,22,39
114,separator&spk,False,2,fp32,False,22,39
115,separator&spk,False,2,bf16-mixed,True,22,39
116,separator&spk,False,2,bf16-mixed,False,22,39
117,separator&spk,False,2,fp16-mixed,True,22,39
118,separator&spk,False,2,fp16-mixed,False,22,39
119,separator&spk,False,4,fp32,True,22,39
120,separator&spk,False,4,fp32,False,22,39


---

## 6. `dynamic` 을 **생략**하면 — 기본값은 `False` 가 아니라 `None` 임

2~5절의 `dynamic` 축은 `True` · `False` 두 값만 흔들었음.
그런데 [torch.compile](https://docs.pytorch.org/docs/stable/generated/torch.compile.html) 의
`dynamic` **기본값은 `None`** 이고, 이것은 두 값 어느 쪽과도 다른 **제3의 동작**임.

| 설정 | 무엇을 하나 |
|---|---|
| `dynamic=False` | 항상 정적으로 특수화. 크기가 바뀌면 **그때마다 재컴파일** |
| `dynamic=True` | 처음부터 **가능한 한 동적**인 커널을 만들어 재컴파일을 피함 |
| **생략 = `None`** | **처음엔 정적으로 잡고, 크기가 바뀌면 그때 동적으로 전환** |

`model.compile()` 처럼 인자를 안 주면 `None` 이 되므로, **실무에서 가장 흔히 쓰는 경로**임에도
2~5절에는 빠져 있었음. 그래서 따로 잼.

측정 범위 — `tf32_lstm=True` 고정, 배치 4가지 × precision 3가지:

| tag | 무엇 | 조건 수 |
|---|---|---|
| `dynamic-none` | `b=16` · fp16 한 점에서 `eager` · `None` · `True` · `False` 를 **같은 GPU 에서 연속** | 4 |
| `dynamic-none-axes` | **`model`** × `b`(2·4·8·16) × precision 3 | 12 |
| `dynamic-none-scopes` | **`separator`** · **`separator&spk`** × 같은 축 | 24 |

`b=1` 은 [BatchNorm](../wesep/models/bsrnn.py) 이 배치 1을 거부해 어떤 방식으로도 실패하므로 뺐음.

In [9]:
rows_none = (bc.run_or_load("dynamic-none")
             + bc.run_or_load("dynamic-none-axes")
             + bc.run_or_load("dynamic-none-scopes"))
display(bc.as_table(rows_none, keep=("scope", "dynamic", "b", "prec")))

,scope,dynamic,b,prec,gpu,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,eager,-,16,fp16-mixed,0,0.0,600.4,599.8,19.38,0,0,0,ok,NaN
1,model,NaN,16,fp16-mixed,0,107.9,549.1,548.9,19.34,5,12,0,ok,NaN
2,model,True,16,fp16-mixed,0,196.8,605.3,606.1,19.40,5,12,0,ok,NaN
3,model,False,16,fp16-mixed,3,107.5,600.2,16976.9,19.34,35,40,22,ok,NaN
4,model,NaN,2,fp32,0,106.5,342.9,351.9,4.95,5,12,0,ok,NaN
5,model,NaN,2,bf16-mixed,0,119.3,360.8,371.5,2.75,5,12,0,ok,NaN
6,model,NaN,2,fp16-mixed,0,117.1,402.5,407.9,2.75,5,12,0,ok,NaN
7,model,NaN,4,fp32,1,111.2,410.7,415.5,9.55,5,12,0,ok,NaN
8,model,NaN,4,bf16-mixed,1,120.4,374.6,382.8,5.11,5,12,0,ok,NaN
9,model,NaN,4,fp16-mixed,1,120.7,353.4,355.6,5.11,5,12,0,ok,NaN


### 6.1 같은 GPU · 같은 조건에서 네 가지를 나란히

`b=16` · `fp16-mixed` 한 점만 뽑아 봄. 조건이 하나뿐이라 **GPU 편차가 끼지 않음.**

In [10]:
one = pd.DataFrame(bc.run_or_load("dynamic-none"))
one["dyn"] = one.dynamic.fillna("None(생략)").astype(str)
one.loc[one.scope == "eager", "dyn"] = "-"
display(one[["scope", "dyn", "compile_s", "ms_step_median", "ms_step_mean",
             "peak_GiB", "graph_break", "dynamo_frames", "status"]])

,scope,dyn,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,status
0,eager,-,0.0,600.4,599.8,19.38,0,0,ok
1,model,None(생략),107.9,549.1,548.9,19.34,5,12,ok
2,model,True,196.8,605.3,606.1,19.40,5,12,ok
3,model,False,107.5,600.2,16976.9,19.34,35,40,ok


### 6.2 `dynamic` 세 값 비교 — scope 별

`speedup` 은 **같은 `(b, prec)` 의 `eager`** 를 기준으로 한 비임.
`ms_per_sample` = `ms_step_median / b` — 배치가 다르면 스텝 시간을 직접 비교할 수 없으므로 이 값으로 봄.

In [11]:
all_rows = rows_base + rows_single + rows_full + rows_none
sel = pd.DataFrame(all_rows)
sel = sel[(sel.status == "ok") & (sel.tf32_lstm == True) & (sel.feature_dim == 128)
          & (sel.num_repeat == 6) & (sel.t == 48000) & (sel.allow_rnn == False)].copy()
sel["dyn"] = sel.dynamic.fillna("None(생략)").astype(str)
sel.loc[sel.scope == "eager", "dyn"] = "-"

g = sel.groupby(["scope", "dyn", "b", "prec"]).agg(
    compile_s=("compile_s", "median"), ms=("ms_step_median", "median")).reset_index()
base = g[g.scope == "eager"][["b", "prec", "ms"]].rename(columns={"ms": "eager_ms"})
g = g.merge(base, on=["b", "prec"], how="left")
g["speedup"] = (g.eager_ms / g.ms).round(3)
g["ms_per_sample"] = (g.ms / g.b).round(1)

# scope x dynamic 별로 — 현행 배치(b=16)에서
display(g[g.b == 16].pivot_table(index=["scope", "dyn"], columns="prec",
                                 values=["compile_s", "ms", "speedup"]).round(2))

compile_s                    ms               speedup  \
prec                   bf16-mixed fp16-mixed bf16-mixed fp16-mixed bf16-mixed   
scope         dyn                                                               
eager         -               0.0       0.00     686.20     624.50       1.00   
model         False         196.4     118.60     630.60     600.20       1.09   
              None(생략)      120.0     112.95     571.70     554.65       1.20   
              True          221.9     214.20     594.10     576.20       1.16   
separator     False           7.8      10.05     586.70     626.75       1.17   
              None(생략)       16.8      17.20     597.60     592.30       1.15   
              True           13.9      16.70     603.90     619.25       1.14   
separator&spk False          34.6      33.95     619.10     618.85       1.11   
              None(생략)       28.9      27.50     589.20     595.80       1.16   
              True           45.6      47.40     611.35     615.90       1.12   

                                   
prec                   fp16-mixed  
scope         dyn                  
eager         -              1.00  
model         False          1.04  
              None(생략)       1.13  
              True           1.08  
separator     False          1.00  
              None(생략)       1.05  
              True           1.01  
separator&spk False          1.01  
              None(생략)       1.05  
              True           1.01

### 6.3 가장 빠른 조합 — 샘플당 시간 기준

**`ms_step_median` 이 작다고 빠른 게 아님** — 배치가 작으면 한 스텝이 처리하는 샘플도 적음.
학습 속도는 `ms_per_sample` 로 봐야 하고, 1 에포크는 그 값 × 27,792 샘플(혼합 13,896 × 화자 2)임.

In [12]:
SAMPLES_PER_EPOCH = 27792
g["epoch_min"] = (g.ms_per_sample * SAMPLES_PER_EPOCH / 1000 / 60).round(1)
g["150ep_h"] = (g.epoch_min * 150 / 60).round(1)

display(g.sort_values("ms_per_sample").head(12)[
    ["scope", "dyn", "b", "prec", "compile_s", "ms", "speedup",
     "ms_per_sample", "epoch_min", "150ep_h"]])

# 현행 설정과의 차이
cur = g[(g.scope == "eager") & (g.b == 16) & (g.prec == "fp16-mixed")]
best = g.sort_values("ms_per_sample").iloc[0]
if len(cur):
    c = cur.iloc[0]
    display(pd.DataFrame([
        {"설정": "현행 (eager · b=16 · fp16-mixed)", "compile_s": c.compile_s,
         "ms_per_sample": c.ms_per_sample, "epoch_min": c.epoch_min, "150ep_h": c["150ep_h"]},
        {"설정": f"최선 ({best.scope} · dynamic={best.dyn} · b={best.b} · {best.prec})",
         "compile_s": best.compile_s, "ms_per_sample": best.ms_per_sample,
         "epoch_min": best.epoch_min, "150ep_h": best["150ep_h"]},
        {"설정": "차이", "compile_s": round(best.compile_s - c.compile_s, 1),
         "ms_per_sample": round(best.ms_per_sample - c.ms_per_sample, 1),
         "epoch_min": round(best.epoch_min - c.epoch_min, 1),
         "150ep_h": round(best["150ep_h"] - c["150ep_h"], 1)},
    ]))

,scope,dyn,b,prec,compile_s,ms,speedup,ms_per_sample,epoch_min,150ep_h
32,model,None(생략),16,fp16-mixed,112.95,554.65,1.126,34.7,16.1,40.2
31,model,None(생략),16,bf16-mixed,120.00,571.70,1.200,35.7,16.5,41.2
43,model,True,16,fp16-mixed,214.20,576.20,1.084,36.0,16.7,41.8
53,separator,False,16,bf16-mixed,7.80,586.70,1.170,36.7,17.0,42.5
97,separator&spk,None(생략),16,bf16-mixed,28.90,589.20,1.165,36.8,17.0,42.5
65,separator,None(생략),16,fp16-mixed,17.20,592.30,1.054,37.0,17.1,42.8
42,model,True,16,bf16-mixed,221.90,594.10,1.155,37.1,17.2,43.0
98,separator&spk,None(생략),16,fp16-mixed,27.50,595.80,1.048,37.2,17.2,43.0
64,separator,None(생략),16,bf16-mixed,16.80,597.60,1.148,37.4,17.3,43.2
21,model,False,16,fp16-mixed,118.60,600.20,1.040,37.5,17.4,43.5


,설정,compile_s,ms_per_sample,epoch_min,150ep_h
0,현행 (eager · b=16 · fp16-mixed),0.00,39.0,18.1,45.2
1,최선 (model · dynamic=None(생략) · b=16 · fp16-mixed),112.95,34.7,16.1,40.2
2,차이,113.00,-4.3,-2.0,-5.0


## 7. 정리 — 무엇을 어떻게 할 것인가

| # | 물음 | 답 |
|---|---|---|
| 1 | 가장 빠른 `scope` 는 | **`model`** (전체 컴파일). 부분 컴파일(`separator` · `separator&spk`)은 eager 와 거의 차이 없음 |
| 2 | `dynamic` 은 무엇으로 | **생략(`None`)** — `True` 보다 컴파일이 절반이고 스텝도 빠름 |
| 3 | 배치가 커질수록 컴파일 이득이 커지나 | 배율은 작은 배치에서 크지만, **샘플당 시간은 `b=16` 이 최선** |
| 4 | `bf16-mixed` 와 `fp16-mixed` | 컴파일 **배율**은 bf16 이 크고(1.20x), **절대 속도**는 fp16 이 빠름 |
| 5 | `tf32_lstm` | fp32 에서만 의미가 있고, 그 fp32 는 `b=16` 에서 **OOM** 이라 쓸 수 없음 |
| 6 | 24 GiB 한 장에서 최대 `b` | **16** (fp16 · bf16). fp32 는 `b=8` 까지 |
| 7 | 웜업 10스텝이 충분했나 | `recompile_in_measure` 가 전부 0 — **충분함** |

### 권고

| 항목 | 지금 | 바꿀 것 |
|---|---|---|
| yaml `batch_size` | 8 (모델 `b=16`) | **그대로** |
| `enable_amp` | true (fp16) | **그대로** |
| 컴파일 | 없음 | **`model.compile()` 한 줄** — `dynamic` 인자를 **주지 말 것** |

거는 자리는 [train.py:235](../wesep/bin/train.py#L235) 의 `model.to(device)` 와 DDP 래핑 사이임.

### 이 노트북이 재지 않는 것

| 항목 | 이유 |
|---|---|
| 모델 구조 변경 | [bsrnn_compile_cost_by_structure.ipynb](bsrnn_compile_cost_by_structure.ipynb) 가 맡음 |
| 정확도·수렴 | 속도만 잼. 컴파일·precision 이 학습 결과를 바꾸는지는 별개 문제임 |
| 실제 데이터 | 로더는 **shape 만 모사**함 — 등록 길이 분포는 근사이고 **미검증**임 (0.1절) |
| 다중 GPU · DDP | 1 GPU 만 봄 |
| **GPU 간 절대 ms 비교** | 조건이 GPU 4장에 나뉘어 감. 6.1절만 **한 GPU 연속 측정**이라 편차가 없음 |